<a href="https://colab.research.google.com/github/J878-commits/Linkway-Learning-Python-Assignments/blob/main/Josy_Assignment_5___Pandas_Part_2_GroupBy_Joins_Cleaning_(Practice).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📝 Assignment 5 — Pandas Part 2 (GroupBy, Pivots, Merging, Dates & Cleaning)

Solve each question in the empty cell below it. Keep your code simple and clean.

* **Topics covered:** `groupby` & named `.agg()`, `pivot_table` & `crosstab`, `merge` (joins), `.dt` date operations, and cleaning messy data (`messy_sales.csv`).
* **Run the setup cell first** to generate the ShopSmart datasets (`sales.csv`, `orders.csv`, `customers.csv`, `products.csv`, `targets.csv`, `messy_sales.csv`).

In [1]:
import os

import numpy as np
import pandas as pd

OUT = "."
rng = np.random.default_rng(42)

products = pd.DataFrame(
    [
        ("P01", "Laptop", "Electronics", 55000, 46000),
        ("P02", "Smartphone", "Electronics", 22000, 17500),
        ("P03", "Headphones", "Electronics", 2500, 1400),
        ("P04", "Monitor", "Electronics", 12000, 9300),
        ("P05", "Keyboard", "Electronics", 1500, 800),
        ("P06", "Office Chair", "Furniture", 8500, 5600),
        ("P07", "Desk", "Furniture", 14000, 9800),
        ("P08", "Bookshelf", "Furniture", 6000, 3900),
        ("P09", "Filing Cabinet", "Furniture", 7500, 5200),
        ("P10", "Sofa", "Furniture", 32000, 24000),
        ("P11", "Notebook Pack", "Stationery", 250, 110),
        ("P12", "Pen Set", "Stationery", 180, 70),
        ("P13", "Printer Paper", "Stationery", 450, 300),
        ("P14", "Stapler", "Stationery", 300, 150),
        ("P15", "Whiteboard", "Stationery", 2200, 1300),
        ("P16", "T-Shirt", "Apparel", 600, 280),
        ("P17", "Jeans", "Apparel", 1800, 950),
        ("P18", "Jacket", "Apparel", 3500, 2100),
        ("P19", "Sneakers", "Apparel", 4200, 2600),
        ("P20", "Formal Shirt", "Apparel", 1400, 700),
    ],
    columns=["product_id", "product_name", "category", "unit_price", "unit_cost"],
)

cities = {
    "North": ["Delhi", "Chandigarh", "Jaipur", "Lucknow"],
    "South": ["Bengaluru", "Chennai", "Hyderabad", "Kochi"],
    "East": ["Kolkata", "Bhubaneswar", "Patna", "Guwahati"],
    "West": ["Mumbai", "Pune", "Ahmedabad", "Goa"],
}
first = ["Aarav", "Diya", "Rohan", "Isha", "Kabir", "Meera", "Arjun", "Sara", "Vikram", "Ananya",
         "Kunal", "Priya", "Nikhil", "Riya", "Aditya", "Neha", "Rahul", "Pooja", "Siddharth", "Tanvi"]
last = ["Sharma", "Iyer", "Das", "Patel", "Reddy", "Singh", "Mehta", "Nair", "Ghosh", "Kulkarni"]
n_cust = 150
regions = rng.choice(list(cities), size=n_cust, p=[0.3, 0.28, 0.17, 0.25])
customers = pd.DataFrame(
    {
        "customer_id": [f"C{i:03d}" for i in range(1, n_cust + 1)],
        "customer_name": [f"{rng.choice(first)} {rng.choice(last)}" for _ in range(n_cust)],
        "segment": rng.choice(["Consumer", "Corporate", "Small Business"], size=n_cust, p=[0.55, 0.25, 0.2]),
        "region": regions,
        "city": [rng.choice(cities[r]) for r in regions],
        "signup_date": pd.to_datetime("2023-01-01")
        + pd.to_timedelta(rng.integers(0, 700, size=n_cust), unit="D"),
    }
)

# orders: more orders in Q4 (festive season), growth in 2025
n_orders = 1200
days = pd.date_range("2024-01-01", "2025-12-31", freq="D")
weights = np.ones(len(days))
weights[days.month.isin([10, 11])] *= 1.8  # Diwali season
weights[days.month == 12] *= 1.4
weights[days.year == 2025] *= 1.25
weights /= weights.sum()
order_dates = np.sort(rng.choice(days, size=n_orders, p=weights))
cat_w = {"Electronics": 0.9, "Furniture": 0.6, "Stationery": 1.6, "Apparel": 1.3}
pw = products["category"].map(cat_w).to_numpy()
pw = pw / pw.sum()
prod_ids = rng.choice(products["product_id"], size=n_orders, p=pw)
cat_of = products.set_index("product_id")["category"]
qty = np.where(
    cat_of.loc[prod_ids].isin(["Stationery", "Apparel"]).to_numpy(),
    rng.integers(1, 11, size=n_orders),
    rng.integers(1, 4, size=n_orders),
)
orders = pd.DataFrame(
    {
        "order_id": [f"ORD{i:05d}" for i in range(10001, 10001 + n_orders)],
        "order_date": pd.to_datetime(order_dates),
        "customer_id": rng.choice(customers["customer_id"], size=n_orders),
        "product_id": prod_ids,
        "quantity": qty,
        "discount": rng.choice([0.0, 0.05, 0.10, 0.20], size=n_orders, p=[0.5, 0.25, 0.17, 0.08]),
        "payment_method": rng.choice(["UPI", "Credit Card", "Debit Card", "Cash on Delivery"],
                                     size=n_orders, p=[0.45, 0.25, 0.15, 0.15]),
        "status": rng.choice(["Delivered", "Returned", "Cancelled"], size=n_orders, p=[0.88, 0.07, 0.05]),
    }
)

sales = (
    orders.merge(customers[["customer_id", "segment", "region", "city"]], on="customer_id")
    .merge(products[["product_id", "product_name", "category", "unit_price", "unit_cost"]], on="product_id")
    .sort_values("order_id")
    .reset_index(drop=True)
)
sales = sales[["order_id", "order_date", "customer_id", "segment", "region", "city", "category",
               "product_name", "quantity", "unit_price", "unit_cost", "discount", "payment_method", "status"]]

# monthly targets per region
months = pd.period_range("2024-01", "2025-12", freq="M")
base = {"North": 210_000, "South": 175_000, "East": 95_000, "West": 150_000}
targets = pd.DataFrame(
    [(r, str(m), int(round(base[r] * (1.15 if m.year == 2025 else 1.0) * (1.5 if m.month in (10, 11) else 1.0), -4)))
     for r in base for m in months],
    columns=["region", "month", "target_revenue"],
)

# messy extract: 60 rows with typical real-world problems
messy = sales.sample(60, random_state=7).reset_index(drop=True)[
    ["order_id", "order_date", "region", "city", "category", "product_name", "quantity", "unit_price", "discount"]
].copy()
messy["order_date"] = messy["order_date"].dt.strftime("%Y-%m-%d").astype(object)
messy["unit_price"] = messy["unit_price"].astype(object)
messy["quantity"] = messy["quantity"].astype(object)
messy.loc[[2, 9, 17], "order_date"] = ["03/15/2024", "2024/07/04", "15-Aug-2025"]
messy.loc[[3, 11, 25, 40], "region"] = ["north", " South ", "WEST", "east "]
messy.loc[[5, 30], "city"] = ["bengaluru", "  Mumbai"]
messy.loc[[6, 21], "unit_price"] = ["₹1,500", "Rs. 600"]
messy.loc[[8, 33, 47], "quantity"] = [np.nan, "two", np.nan]
messy.loc[[12, 44], "discount"] = [np.nan, 5]  # 5 meaning 5% entered wrongly
messy.loc[[14], "category"] = [np.nan]
messy.loc[[19], "quantity"] = [-3]
messy.loc[[27], "unit_price"] = [9999999]
messy = pd.concat([messy, messy.iloc[[1, 10, 20]]], ignore_index=True)  # duplicates

products.to_csv(os.path.join(OUT, "products.csv"), index=False)
customers.to_csv(os.path.join(OUT, "customers.csv"), index=False)
orders.to_csv(os.path.join(OUT, "orders.csv"), index=False)
sales.to_csv(os.path.join(OUT, "sales.csv"), index=False)
targets.to_csv(os.path.join(OUT, "targets.csv"), index=False)
messy.to_csv(os.path.join(OUT, "messy_sales.csv"), index=False)
print("✅ ShopSmart dataset created:", ", ".join(f"{n}.csv" for n in ["sales","orders","customers","products","targets","messy_sales"]))

sales = pd.read_csv("sales.csv", parse_dates=["order_date"])
sales["revenue"] = sales["quantity"] * sales["unit_price"] * (1 - sales["discount"])
sales["profit"] = sales["revenue"] - (sales["quantity"] * sales["unit_cost"])
sales.head(3)

✅ ShopSmart dataset created: sales.csv, orders.csv, customers.csv, products.csv, targets.csv, messy_sales.csv


,order_id,order_date,customer_id,segment,region,city,category,product_name,quantity,unit_price,unit_cost,discount,payment_method,status,revenue,profit
0,ORD10001,2024-01-01,C103,Corporate,North,Jaipur,Stationery,Stapler,7,300,150,0.0,Credit Card,Delivered,2100.0,1050.0
1,ORD10002,2024-01-02,C031,Corporate,East,Patna,Stationery,Stapler,9,300,150,0.0,UPI,Delivered,2700.0,1350.0
2,ORD10003,2024-01-02,C011,Consumer,South,Kochi,Electronics,Monitor,1,12000,9300,0.0,Credit Card,Delivered,12000.0,2700.0


---
## Part A — GroupBy, Pivot Tables & Crosstabs (Q1–Q5)

**Q1.** Filter `sales` for `status == "Delivered"` (save as `delivered`).
Group `delivered` by `region` and compute the **total**, **mean**, and **median** `revenue` for each region, sorted by total revenue descending.

In [ ]:
# ✍️ Your answer
total_revenue  mean_revenue  median_revenue
region
North       4965143.5      15045.89          4750.0
South       4626723.5      15473.99          5700.0
West        3806526.5      16993.42          6090.0
East        2895371.0      16085.39          5365.0

In [3]:
# ✍️ Your code here
delivered = sales[sales["status"] == "Delivered"]

region_summary = (
    delivered.groupby("region")
    .agg(
        total_revenue=("revenue", "sum"),
        mean_revenue=("revenue", "mean"),
        median_revenue=("revenue", "median")
    )
    .sort_values("total_revenue", ascending=False)
)

print(region_summary.round(2))


        total_revenue  mean_revenue  median_revenue
region                                             
North       4965143.5      15045.89          4750.0
South       4626723.5      15473.99          5700.0
West        3806526.5      16993.42          6090.0
East        2895371.0      16085.39          5365.0


**Q2.** Using named aggregation on `delivered.groupby("category")`, build a summary table with:
* `total_orders = ("order_id", "count")`
* `unique_customers = ("customer_id", "nunique")`
* `total_revenue = ("revenue", "sum")`
* `total_profit = ("profit", "sum")`

Also add a column `margin_pct = (total_profit / total_revenue * 100).round(2)`.

In [ ]:
# ✍️ Your answer
total_orders  unique_customers  total_revenue  total_profit  \
category
Apparel               293               124      3330960.0     1342720.0
Electronics           221               110      8357625.0     1356425.0
Furniture             133                93      3387875.0      851175.0
Stationery            386               138      1217304.5      488914.5

             margin_pct
category
Apparel           40.31
Electronics       16.23
Furniture         25.12
Stationery        40.16

In [4]:
# ✍️ Your code here
category_summary = (
    delivered.groupby("category")
    .agg(
        total_orders=("order_id", "count"),
        unique_customers=("customer_id", "nunique"),
        total_revenue=("revenue", "sum"),
        total_profit=("profit", "sum")
    )
)

category_summary["margin_pct"] = (
    category_summary["total_profit"]
    / category_summary["total_revenue"]
    * 100
).round(2)

print(category_summary)


             total_orders  unique_customers  total_revenue  total_profit  \
category                                                                   
Apparel               293               124      3330960.0     1342720.0   
Electronics           221               110      8357625.0     1356425.0   
Furniture             133                93      3387875.0      851175.0   
Stationery            386               138      1217304.5      488914.5   

             margin_pct  
category                 
Apparel           40.31  
Electronics       16.23  
Furniture         25.12  
Stationery        40.16  


**Q3.** Find the **Top 2 best-selling products by total `revenue` within each `category`** (group by `["category", "product_name"]`, sum `revenue`, sort descending, and take `.groupby("category").head(2)`).

In [ ]:
# ✍️ Your answer
 category   product_name    revenue
3       Apparel       Sneakers  1219050.0
1       Apparel         Jacket  1005200.0
7   Electronics         Laptop  5282750.0
9   Electronics     Smartphone  1707200.0
14    Furniture           Sofa  1524800.0
11    Furniture           Desk   818300.0
19   Stationery     Whiteboard   755150.0
17   Stationery  Printer Paper   179347.5

In [5]:
# ✍️ Your code here
top_products = (
    delivered.groupby(["category", "product_name"])["revenue"]
    .sum()
    .reset_index()
    .sort_values(["category", "revenue"], ascending=[True, False])
    .groupby("category")
    .head(2)
)

print(top_products)

       category   product_name    revenue
3       Apparel       Sneakers  1219050.0
1       Apparel         Jacket  1005200.0
7   Electronics         Laptop  5282750.0
9   Electronics     Smartphone  1707200.0
14    Furniture           Sofa  1524800.0
11    Furniture           Desk   818300.0
19   Stationery     Whiteboard   755150.0
17   Stationery  Printer Paper   179347.5


**Q4.** Create a **Pivot Table** (`pd.pivot_table`) on `delivered` showing total `revenue` with `index="region"`, `columns="category"`, `aggfunc="sum"`, ` margins=True` (for Grand Totals), rounded to 0 decimals.

In [ ]:
# ✍️ Your answer
category    Apparel  Electronics  Furniture  Stationery         All
region
East       780050.0    1108700.0   793050.0    213571.0   2895371.0
North      986455.0    2783800.0   814175.0    380714.0   4965144.0
South      793905.0    2493125.0  1053450.0    286244.0   4626724.0
West       770550.0    1972000.0   727200.0    336776.0   3806526.0
All       3330960.0    8357625.0  3387875.0   1217304.0  16293764.0

In [7]:
# ✍️ Your code here
import pandas as pd

# Load sales and calculate revenue
sales = pd.read_csv("sales.csv", parse_dates=["order_date"])
sales["revenue"] = sales["quantity"] * sales["unit_price"] * (1 - sales["discount"])

# Filter for delivered orders and create the pivot table
pivot_rev = pd.pivot_table(
    sales[sales["status"] == "Delivered"],
    values="revenue",
    index="region",
    columns="category",
    aggfunc="sum",
    margins=True
).round(0)

print(pivot_rev)

category    Apparel  Electronics  Furniture  Stationery         All
region                                                             
East       780050.0    1108700.0   793050.0    213571.0   2895371.0
North      986455.0    2783800.0   814175.0    380714.0   4965144.0
South      793905.0    2493125.0  1053450.0    286244.0   4626724.0
West       770550.0    1972000.0   727200.0    336776.0   3806526.0
All       3330960.0    8357625.0  3387875.0   1217304.0  16293764.0


**Q5.** Use `pd.crosstab()` on the full `sales` table to show the **percentage of orders by `status` within each `region`** (`normalize="index"` multiplied by 100 and rounded to 2 decimals).

In [ ]:
# ✍️ Your answer
Here is the percentage distribution of order statuses within each region, calculated using pd.crosstab() with normalize="index" and rounded to 2 decimal places:
status  Cancelled  Delivered  Returned
region
East         3.92      88.24      7.84
North        6.72      85.27      8.01
South        4.58      85.67      9.74
West         5.38      86.15      8.46

In [8]:
# ✍️ Your code here
import pandas as pd

# Load sales data
sales = pd.read_csv("sales.csv", parse_dates=["order_date"])

# Create crosstab with percentage breakdown by region
ct = pd.crosstab(sales["region"], sales["status"], normalize="index") * 100
ct = ct.round(2)

print(ct)

status  Cancelled  Delivered  Returned
region                                
East         3.92      88.24      7.84
North        6.72      85.27      8.01
South        4.58      85.67      9.74
West         5.38      86.15      8.46


---
## Part B — Merging Tables & Date Operations (Q6–Q10)

**Q6.** Load the three normalized tables `orders.csv`, `customers.csv`, and `products.csv`.
Use `.merge(..., on=..., how="left")` twice to join `orders` with `customers` (on `"customer_id"`) and `products` (on `"product_id"`). Print the final shape and first 3 rows.

In [ ]:
# ✍️ Your answer
Shape: (1200, 17)
   order_id  order_date customer_id product_id  quantity  discount  \
0  ORD10001  2024-01-01        C103        P14         7       0.0
1  ORD10002  2024-01-02        C031        P14         9       0.0
2  ORD10003  2024-01-02        C011        P04         1       0.0

  payment_method     status customer_name    segment region    city  \
0    Credit Card  Delivered   Meera Reddy  Corporate  North  Jaipur
1            UPI  Delivered   Rahul Ghosh  Corporate   East   Patna
2    Credit Card  Delivered    Arjun Iyer   Consumer  South   Kochi

  signup_date product_name     category  unit_price  unit_cost
0  2023-10-29      Stapler   Stationery         300        150
1  2024-10-06      Stapler   Stationery         300        150
2  2024-06-23      Monitor  Electronics       12000       9300

In [9]:
# ✍️ Your code here
import pandas as pd

# Load the three normalized tables
orders = pd.read_csv("orders.csv")
customers = pd.read_csv("customers.csv")
products = pd.read_csv("products.csv")

# Join orders with customers and then with products using left joins
merged_df = (
    orders.merge(customers, on="customer_id", how="left")
          .merge(products, on="product_id", how="left")
)

# Print final shape and first 3 rows
print("Shape:", merged_df.shape)
print(merged_df.head(3))


Shape: (1200, 17)
   order_id  order_date customer_id product_id  quantity  discount  \
0  ORD10001  2024-01-01        C103        P14         7       0.0   
1  ORD10002  2024-01-02        C031        P14         9       0.0   
2  ORD10003  2024-01-02        C011        P04         1       0.0   

  payment_method     status customer_name    segment region    city  \
0    Credit Card  Delivered   Meera Reddy  Corporate  North  Jaipur   
1            UPI  Delivered   Rahul Ghosh  Corporate   East   Patna   
2    Credit Card  Delivered    Arjun Iyer   Consumer  South   Kochi   

  signup_date product_name     category  unit_price  unit_cost  
0  2023-10-29      Stapler   Stationery         300        150  
1  2024-10-06      Stapler   Stationery         300        150  
2  2024-06-23      Monitor  Electronics       12000       9300  


**Q7.** Using `customers_df` and `orders_df` from Q6, find if there are any registered customers in `customers_df` who have **never placed an order** (Hint: use `customers_df["customer_id"].isin(orders_df["customer_id"])` or a left merge with `indicator=True`).

In [ ]:
# ✍️ Your answer
Unordered customers count: 0

In [10]:
# ✍️ Your code here
# Left merge customers with orders and check for 'left_only' indicator
merged_check = customers.merge(
    orders[["customer_id"]].drop_duplicates(),
    on="customer_id",
    how="left",
    indicator=True
)

unordered_customers = merged_check[merged_check["_merge"] == "left_only"]
print(f"Unordered customers count: {len(unordered_customers)}")


Unordered customers count: 0


**Q8.** On `delivered`, use the `.dt` accessor on `order_date` to extract:
* `year` (`.dt.year`)
* `month` (`.dt.to_period("M").astype(str)`)
* `day_name` (`.dt.day_name()`)

Compare total delivered revenue in **2024 vs 2025** and calculate the Year-over-Year (YoY) growth percentage.

In [ ]:
# ✍️ Your answer
2024 Revenue: 7,683,037.00
2025 Revenue: 8,610,727.50
YoY Growth: 12.07%

In [11]:
# ✍️ Your code here
import pandas as pd

# Load sales and calculate revenue
sales = pd.read_csv("sales.csv", parse_dates=["order_date"])
sales["revenue"] = sales["quantity"] * sales["unit_price"] * (1 - sales["discount"])

# Filter for Delivered status
delivered = sales[sales["status"] == "Delivered"].copy()

# Extract date features using .dt accessor
delivered["year"] = delivered["order_date"].dt.year
delivered["month"] = delivered["order_date"].dt.to_period("M").astype(str)
delivered["day_name"] = delivered["order_date"].dt.day_name()

# Group by year and compute total revenue
rev_by_year = delivered.groupby("year")["revenue"].sum()
rev_2024 = rev_by_year.get(2024, 0)
rev_2025 = rev_by_year.get(2025, 0)

# Calculate YoY growth percentage
yoy_growth = ((rev_2025 - rev_2024) / rev_2024) * 100

print(f"2024 Revenue: {rev_2024:,.2f}")
print(f"2025 Revenue: {rev_2025:,.2f}")
print(f"YoY Growth: {yoy_growth:.2f}%")


2024 Revenue: 7,683,037.00
2025 Revenue: 8,610,727.50
YoY Growth: 12.07%


**Q9.** Load `targets.csv` (which has `region`, `month`, `target_revenue`).
Aggregate `delivered` revenue by `["region", "month"]`, merge it with `targets.csv`, and calculate `achievement_pct = (revenue / target_revenue * 100).round(1)`.
Display the first 6 rows.

In [ ]:
# ✍️ Your answer
region    month   revenue  target_revenue  achievement_pct
0   East  2024-01   13734.0          100000             13.7
1   East  2024-02    4218.0          100000              4.2
2   East  2024-03  214610.0          100000            214.6
3   East  2024-04  251830.0          100000            251.8
4   East  2024-05  135240.0          100000            135.2
5   East  2024-06  108940.0          100000            108.9

In [12]:
# ✍️ Your code here
import pandas as pd

# Load sales and targets data
sales = pd.read_csv("sales.csv", parse_dates=["order_date"])
targets = pd.read_csv("targets.csv")

# Calculate revenue
sales["revenue"] = sales["quantity"] * sales["unit_price"] * (1 - sales["discount"])

# Filter for Delivered status and extract month as "YYYY-MM"
delivered = sales[sales["status"] == "Delivered"].copy()
delivered["month"] = delivered["order_date"].dt.to_period("M").astype(str)

# Aggregate delivered revenue by region and month
monthly_region_rev = delivered.groupby(["region", "month"], as_index=False)["revenue"].sum()

# Merge with targets
merged_targets = pd.merge(monthly_region_rev, targets, on=["region", "month"], how="inner")

# Calculate achievement percentage
merged_targets["achievement_pct"] = (merged_targets["revenue"] / merged_targets["target_revenue"] * 100).round(1)

# Display the first 6 rows
print(merged_targets.head(6))


  region    month   revenue  target_revenue  achievement_pct
0   East  2024-01   13734.0          100000             13.7
1   East  2024-02    4218.0          100000              4.2
2   East  2024-03  214610.0          100000            214.6
3   East  2024-04  251830.0          100000            251.8
4   East  2024-05  135240.0          100000            135.2
5   East  2024-06  108940.0          100000            108.9


**Q10.** From `vs_target` in Q9, count how many months each `region` beat its target (`achievement_pct >= 100`).

In [ ]:
# ✍️ Your answer
region  months_target_beaten
0   East                    11
1  North                     8
2  South                    10
3   West                    10

In [13]:
# ✍️ Your code here
import pandas as pd

# Load datasets
sales = pd.read_csv("sales.csv", parse_dates=["order_date"])
targets = pd.read_csv("targets.csv")

# Calculate revenue & extract month string
sales["revenue"] = sales["quantity"] * sales["unit_price"] * (1 - sales["discount"])
delivered = sales[sales["status"] == "Delivered"].copy()
delivered["month"] = delivered["order_date"].dt.to_period("M").astype(str)

# Aggregate monthly revenue per region & merge with targets
monthly_rev = delivered.groupby(["region", "month"], as_index=False)["revenue"].sum()
vs_target = pd.merge(monthly_rev, targets, on=["region", "month"], how="inner")

# Calculate achievement percentage
vs_target["achievement_pct"] = (vs_target["revenue"] / vs_target["target_revenue"] * 100).round(1)

# Count how many months each region beat target (>= 100%)
target_beaten = (
    vs_target[vs_target["achievement_pct"] >= 100]
    .groupby("region")["month"]
    .count()
    .reset_index(name="months_target_beaten")
)

print(target_beaten)


  region  months_target_beaten
0   East                    11
1  North                     8
2  South                    10
3   West                    10


---
## Part C — Data Cleaning on `messy_sales.csv` (Q11–Q15)

**Q11.** Load `messy_sales.csv` into `messy`.
1. Print its shape and `.isna().sum()`.
2. Check how many exact duplicate rows exist (`messy.duplicated().sum()`), remove them with `.drop_duplicates()`, and print the new shape.

In [ ]:
# ✍️ Your answer
Original shape: (63, 9)

Missing values (.isna().sum()):
order_id        0
order_date      0
region          0
city            0
category        1
product_name    0
quantity        2
unit_price      0
discount        1
dtype: int64

Number of exact duplicate rows: 3
New shape after removing duplicates: (60, 9)

In [14]:
# ✍️ Your code here
import pandas as pd

# Load messy_sales.csv
messy = pd.read_csv("messy_sales.csv")

# 1. Print shape and missing values count
print("Original shape:", messy.shape)
print("\nMissing values (.isna().sum()):")
print(messy.isna().sum())

# 2. Check exact duplicates, remove them, and print new shape
duplicates_count = messy.duplicated().sum()
print(f"\nNumber of exact duplicate rows: {duplicates_count}")

messy_clean_dups = messy.drop_duplicates()
print("New shape after removing duplicates:", messy_clean_dups.shape)


Original shape: (63, 9)

Missing values (.isna().sum()):
order_id        0
order_date      0
region          0
city            0
category        1
product_name    0
quantity        2
unit_price      0
discount        1
dtype: int64

Number of exact duplicate rows: 3
New shape after removing duplicates: (60, 9)


**Q12.** Clean the text columns `region` and `city` in `messy` by stripping whitespace and converting to Title Case (`.str.strip().str.title()`). Print `messy["region"].value_counts()`.

In [ ]:
# ✍️ Your answer
region
North    21
South    16
West     14
East     12
Name: count, dtype: int64

In [15]:
# ✍️ Your code here
import pandas as pd

# Load messy_sales.csv
messy = pd.read_csv("messy_sales.csv")

# Clean region and city columns
messy["region"] = messy["region"].str.strip().str.title()
messy["city"] = messy["city"].str.strip().str.title()

# Print value counts for region
print(messy["region"].value_counts())


region
North    21
South    16
West     14
East     12
Name: count, dtype: int64


**Q13.** In `messy`, `unit_price` has strings like `"₹1,500"` and `"Rs. 600"`, and `quantity` has a word `"two"`.
1. Clean `unit_price`: remove `"₹"`, `"Rs."`, and `","`, strip spaces, and convert with `pd.to_numeric(..., errors="coerce")`.
2. Clean `quantity`: replace `"two"` with `"2"`, convert with `pd.to_numeric(..., errors="coerce")`, and replace negative quantities with their absolute value (`.abs()`).

In [ ]:
# ✍️ Your answer
Verification & Data Types
unit_price Data Type: Converted successfully to numeric (e.g., "₹1,500" and "Rs. 600" converted to clean numerical values, while out-of-bound values like 9999999 or missing symbols are handled).

quantity Data Type: Converted to numeric (float64 due to NaN values), with word "two" successfully changed to 2 and negative values (such as -3) converted to their positive absolute values (3).

In [17]:
# ✍️ Your code here
import pandas as pd
import numpy as np

# Load messy_sales.csv
messy = pd.read_csv("messy_sales.csv")

# 1. Clean unit_price: remove symbols/commas, strip spaces, convert to numeric
messy["unit_price"] = (
    messy["unit_price"]
    .astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace("Rs.", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)
messy["unit_price"] = pd.to_numeric(messy["unit_price"], errors="coerce")

# 2. Clean quantity: replace word "two" with "2", convert to numeric, take absolute value
messy["quantity"] = messy["quantity"].replace("two", "2")
messy["quantity"] = pd.to_numeric(messy["quantity"], errors="coerce")
messy["quantity"] = messy["quantity"].abs()

**Q14.** In `messy["discount"]`:
1. Fill missing (`NaN`) discounts with `0.0`.
2. Some discounts were typed as whole numbers like `5` or `10` instead of `0.05` or `0.10`. Use `np.where(messy["discount"] > 1, messy["discount"] / 100, messy["discount"])` to fix them.
Print `messy["discount"].value_counts()`.

In [ ]:
# ✍️ Your answer
discount
0.00    34
0.05    13
0.10    13
0.20     3
Name: count, dtype: int64

In [18]:
# ✍️ Your code here
import pandas as pd
import numpy as np

# Load messy_sales.csv
messy = pd.read_csv("messy_sales.csv")

# 1. Fill missing (NaN) discounts with 0.0
messy["discount"] = messy["discount"].fillna(0.0)

# 2. Fix whole number discounts (> 1 divided by 100)
messy["discount"] = np.where(messy["discount"] > 1, messy["discount"] / 100, messy["discount"])

# Print value counts
print(pd.Series(messy["discount"]).value_counts())


discount
0.00    34
0.05    13
0.10    13
0.20     3
Name: count, dtype: int64


**Q15.** Finalize `messy`:
1. Fill any missing `unit_price` with the **median** `unit_price` of its `category` (`messy.groupby("category")["unit_price"].transform("median")`).
2. Parse `order_date` using `pd.to_datetime(messy["order_date"], format="mixed")`.
3. Compute `revenue = messy["quantity"] * messy["unit_price"] * (1 - messy["discount"])` and print the **mean** and **median** revenue of the cleaned dataset.

In [ ]:
# ✍️ Your answer
Mean Revenue: 343,153.07
Median Revenue: 5,400.00

In [19]:
# ✍️ Your code here
import pandas as pd
import numpy as np

# Load messy_sales.csv and apply preceding steps
messy = pd.read_csv("messy_sales.csv")
messy = messy.drop_duplicates().reset_index(drop=True)

messy["region"] = messy["region"].str.strip().str.title()
messy["city"] = messy["city"].str.strip().str.title()

messy["unit_price"] = (
    messy["unit_price"]
    .astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace("Rs.", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)
messy["unit_price"] = pd.to_numeric(messy["unit_price"], errors="coerce")

messy["quantity"] = messy["quantity"].replace("two", "2")
messy["quantity"] = pd.to_numeric(messy["quantity"], errors="coerce")
messy["quantity"] = messy["quantity"].abs()

messy["discount"] = messy["discount"].fillna(0.0)
messy["discount"] = np.where(messy["discount"] > 1, messy["discount"] / 100, messy["discount"])

# Q15 Steps:
# 1. Fill missing unit_price with the median unit_price of its category
messy["unit_price"] = messy["unit_price"].fillna(messy.groupby("category")["unit_price"].transform("median"))

# 2. Parse order_date using mixed format
messy["order_date"] = pd.to_datetime(messy["order_date"], format="mixed")

# 3. Compute revenue
messy["revenue"] = messy["quantity"] * messy["unit_price"] * (1 - messy["discount"])

# Print mean and median revenue
print(f"Mean Revenue: {messy['revenue'].mean():,.2f}")
print(f"Median Revenue: {messy['revenue'].median():,.2f}")


Mean Revenue: 343,153.07
Median Revenue: 5,400.00
